<a href="https://colab.research.google.com/github/NathanPintoNascimento/enem-medallion-etl/blob/main/Projeto_ENEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Rodando na nuvem, não no meu PC 🎉")
import sys
print(sys.version)

Rodando na nuvem, não no meu PC 🎉
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_PATH = '/content/drive/MyDrive/enem-medallion-etl'
os.makedirs(f'{BASE_PATH}/data/raw', exist_ok=True)
os.makedirs(f'{BASE_PATH}/data/bronze', exist_ok=True)
os.makedirs(f'{BASE_PATH}/data/silver', exist_ok=True)
os.makedirs(f'{BASE_PATH}/data/gold', exist_ok=True)

print("Pastas criadas em:", BASE_PATH)

Mounted at /content/drive
Pastas criadas em: /content/drive/MyDrive/enem-medallion-etl


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

np.random.seed(42)

def gerar_dados_sinteticos_enem(n=50_000, ano=2023):
    """Simula a estrutura real dos microdados do ENEM (INEP)."""

    ufs = ['SP','RJ','MG','BA','RS','PR','PE','CE','PA','SC',
           'GO','MA','PB','ES','RN','AL','PI','MT','MS','DF',
           'SE','RO','TO','AC','AM','AP','RR']

    tipo_escola = np.random.choice([1, 2, 3], size=n, p=[0.15, 0.75, 0.10])
    # 1 = Federal, 2 = Estadual/Municipal (pública), 3 = Privada
    # (INEP na real usa outro código — vamos documentar isso na Silver)

    q006 = np.random.choice(list('ABCDEFGHIJKLMNOPQ'), size=n)
    # Q006 = renda familiar, de "Nenhuma renda" (A) até "Mais de 20 salários" (Q)

    df = pd.DataFrame({
        'NU_INSCRICAO': [f'{ano}{i:012d}' for i in range(n)],
        'NU_ANO': ano,
        'SG_UF_PROVA': np.random.choice(ufs, size=n),
        'TP_ESCOLA': tipo_escola,
        'Q006': q006,
        'NU_NOTA_CN': np.round(np.random.normal(520, 90, n).clip(0, 1000), 1),
        'NU_NOTA_CH': np.round(np.random.normal(540, 85, n).clip(0, 1000), 1),
        'NU_NOTA_LC': np.round(np.random.normal(530, 80, n).clip(0, 1000), 1),
        'NU_NOTA_MT': np.round(np.random.normal(510, 100, n).clip(0, 1000), 1),
        'NU_NOTA_REDACAO': np.round(np.random.normal(600, 150, n).clip(0, 1000), 1),
        'TP_STATUS_REDACAO': np.random.choice([1, 2, 3, 4], size=n, p=[0.9, 0.05, 0.03, 0.02]),
        # 1 = Sem problemas, outros = ausente/eliminado/etc
    })
    return df

df_raw = gerar_dados_sinteticos_enem(n=50_000, ano=2023)
df_raw.head()

,NU_INSCRICAO,NU_ANO,SG_UF_PROVA,TP_ESCOLA,Q006,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,TP_STATUS_REDACAO
0,2023000000000000,2023,SP,2,I,706.5,671.6,454.5,470.1,663.2,1
1,2023000000000001,2023,RR,3,D,680.7,439.1,546.0,588.8,494.8,1
2,2023000000000002,2023,RS,2,F,500.1,488.9,759.9,704.4,477.7,1
3,2023000000000003,2023,PI,2,B,410.5,516.6,620.6,380.2,571.6,3
4,2023000000000004,2023,AP,2,A,503.9,445.6,659.2,507.7,643.7,1


In [ ]:
import hashlib

def ingestao_bronze(df: pd.DataFrame, fonte: str = 'sintetico') -> pd.DataFrame:
    """
    Camada Bronze: preserva os dados como estão, só como String,
    e adiciona colunas de auditoria.
    """
    df_bronze = df.copy()

    # Regra da Bronze: tudo vira string, sem interpretar tipo ainda
    for col in df_bronze.columns:
        df_bronze[col] = df_bronze[col].astype(str)

    # Colunas de auditoria
    df_bronze['_source'] = fonte
    df_bronze['_ingested_at'] = datetime.now().isoformat()

    # Hash simples do conteúdo, pra rastrear se o dado mudou entre execuções
    conteudo = pd.util.hash_pandas_object(df, index=False).values.tobytes()
    df_bronze['_file_hash'] = hashlib.md5(conteudo).hexdigest()

    return df_bronze


df_bronze = ingestao_bronze(df_raw, fonte='sintetico_dev')

# Salva particionado por ano (aqui é só uma pasta por ano, dentro do Drive)
ano = df_raw['NU_ANO'].iloc[0]
caminho_bronze = f'{BASE_PATH}/data/bronze/ano={ano}'
os.makedirs(caminho_bronze, exist_ok=True)
df_bronze.to_parquet(f'{caminho_bronze}/bronze.parquet', index=False)

print(f"Bronze salva em: {caminho_bronze}")
print(f"Linhas: {len(df_bronze)}")
df_bronze.head(3)

Bronze salva em: /content/drive/MyDrive/enem-medallion-etl/data/bronze/ano=2023
Linhas: 50000


,NU_INSCRICAO,NU_ANO,SG_UF_PROVA,TP_ESCOLA,Q006,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,TP_STATUS_REDACAO,_source,_ingested_at,_file_hash
0,2023000000000000,2023,SP,2,I,706.5,671.6,454.5,470.1,663.2,1,sintetico_dev,2026-07-13T14:42:01.260150,17f5c5582c61e656d0cdf465dc3aca2b
1,2023000000000001,2023,RR,3,D,680.7,439.1,546.0,588.8,494.8,1,sintetico_dev,2026-07-13T14:42:01.260150,17f5c5582c61e656d0cdf465dc3aca2b
2,2023000000000002,2023,RS,2,F,500.1,488.9,759.9,704.4,477.7,1,sintetico_dev,2026-07-13T14:42:01.260150,17f5c5582c61e656d0cdf465dc3aca2b


In [ ]:
# Dicionário oficial do INEP para Q006 (renda familiar mensal)
DICT_Q006 = {
    'A': 'Nenhuma renda',
    'B': 'Até R$ 1.320,00',
    'C': 'De R$ 1.320,01 até R$ 1.980,00',
    'D': 'De R$ 1.980,01 até R$ 2.640,00',
    'E': 'De R$ 2.640,01 até R$ 3.300,00',
    'F': 'De R$ 3.300,01 até R$ 3.960,00',
    'G': 'De R$ 3.960,01 até R$ 5.280,00',
    'H': 'De R$ 5.280,01 até R$ 6.600,00',
    'I': 'De R$ 6.600,01 até R$ 7.920,00',
    'J': 'De R$ 7.920,01 até R$ 9.240,00',
    'K': 'De R$ 9.240,01 até R$ 10.560,00',
    'L': 'De R$ 10.560,01 até R$ 11.880,00',
    'M': 'De R$ 11.880,01 até R$ 13.200,00',
    'N': 'De R$ 13.200,01 até R$ 15.840,00',
    'O': 'De R$ 15.840,01 até R$ 19.800,00',
    'P': 'De R$ 19.800,01 até R$ 26.400,00',
    'Q': 'Mais de R$ 26.400,00',
}

# Faixas agregadas — pra análise ficar legível (ninguém quer gráfico com 17 categorias)
def classificar_faixa_renda(letra_q006: str) -> str:
    if letra_q006 in ['A', 'B', 'C']:
        return 'Baixa renda'
    elif letra_q006 in ['D', 'E', 'F', 'G']:
        return 'Média-baixa renda'
    elif letra_q006 in ['H', 'I', 'J', 'K']:
        return 'Média renda'
    elif letra_q006 in ['L', 'M', 'N']:
        return 'Média-alta renda'
    elif letra_q006 in ['O', 'P', 'Q']:
        return 'Alta renda'
    else:
        return 'Não informado'

print(DICT_Q006['I'], '→', classificar_faixa_renda('I'))

De R$ 6.600,01 até R$ 7.920,00 → Média renda


In [6]:
# De-para de nomes de colunas (INEP → snake_case legível)
RENOMEAR_COLUNAS = {
    'NU_INSCRICAO': 'numero_inscricao',
    'NU_ANO': 'ano',
    'SG_UF_PROVA': 'uf',
    'TP_ESCOLA': 'tipo_escola_codigo',
    'Q006': 'renda_codigo',
    'NU_NOTA_CN': 'nota_ciencias_natureza',
    'NU_NOTA_CH': 'nota_ciencias_humanas',
    'NU_NOTA_LC': 'nota_linguagens',
    'NU_NOTA_MT': 'nota_matematica',
    'NU_NOTA_REDACAO': 'nota_redacao',
    'TP_STATUS_REDACAO': 'status_redacao_codigo',
}

# Mapa de regiões por UF (não vem pronto no ENEM, precisamos derivar)
REGIAO_POR_UF = {
    'AC':'Norte','AP':'Norte','AM':'Norte','PA':'Norte','RO':'Norte','RR':'Norte','TO':'Norte',
    'AL':'Nordeste','BA':'Nordeste','CE':'Nordeste','MA':'Nordeste','PB':'Nordeste',
    'PE':'Nordeste','PI':'Nordeste','RN':'Nordeste','SE':'Nordeste',
    'DF':'Centro-Oeste','GO':'Centro-Oeste','MT':'Centro-Oeste','MS':'Centro-Oeste',
    'ES':'Sudeste','MG':'Sudeste','RJ':'Sudeste','SP':'Sudeste',
    'PR':'Sul','RS':'Sul','SC':'Sul',
}

def transformar_silver(df_bronze: pd.DataFrame) -> pd.DataFrame:
    """Camada Silver: limpa, tipa e enriquece os dados da Bronze."""

    df = df_bronze.copy()

    # 1. Renomear colunas
    df = df.rename(columns=RENOMEAR_COLUNAS)

    # 2. Tipagem correta (na Bronze estava tudo string)
    colunas_notas = ['nota_ciencias_natureza', 'nota_ciencias_humanas',
                      'nota_linguagens', 'nota_matematica', 'nota_redacao']
    for col in colunas_notas:
        df[col] = pd.to_numeric(df[col], errors='coerce')  # vira NaN se não for número válido

    df['ano'] = pd.to_numeric(df['ano'], errors='coerce').astype('Int64')
    df['tipo_escola_codigo'] = pd.to_numeric(df['tipo_escola_codigo'], errors='coerce').astype('Int64')
    df['status_redacao_codigo'] = pd.to_numeric(df['status_redacao_codigo'], errors='coerce').astype('Int64')

    linhas_antes = len(df)

    # 3. Filtrar registros inválidos:
    #    - notas nulas (não fez a prova / não converteu)
    #    - redação "eliminada" (status != 1, ou seja, teve algum problema)
    df = df.dropna(subset=colunas_notas)
    df = df[df['status_redacao_codigo'] == 1]

    # 4. Deduplicar por inscrição (garantia — não deveria ter duplicata, mas defensivo)
    df = df.drop_duplicates(subset=['numero_inscricao'])

    linhas_depois = len(df)
    pct_retido = round(100 * linhas_depois / linhas_antes, 1)
    print(f"Silver: {linhas_antes} → {linhas_depois} linhas ({pct_retido}% retido)")

    # 5. Colunas derivadas
    df['faixa_renda'] = df['renda_codigo'].apply(classificar_faixa_renda)
    df['renda_descricao'] = df['renda_codigo'].map(DICT_Q006)

    df['tipo_escola'] = df['tipo_escola_codigo'].map({1: 'Pública', 2: 'Pública', 3: 'Privada'})
    # (1=Federal e 2=Estadual/Municipal contam como pública; 3=Privada)

    df['regiao'] = df['uf'].map(REGIAO_POR_UF)

    return df


df_silver = transformar_silver(df_bronze)

caminho_silver = f'{BASE_PATH}/data/silver/ano={ano}'
os.makedirs(caminho_silver, exist_ok=True)
df_silver.to_parquet(f'{caminho_silver}/silver.parquet', index=False)

print(f"Silver salva em: {caminho_silver}")
df_silver[['numero_inscricao','uf','regiao','tipo_escola','faixa_renda','nota_matematica']].head()

Silver: 50000 → 45041 linhas (90.1% retido)
Silver salva em: /content/drive/MyDrive/enem-medallion-etl/data/silver/ano=2023


,numero_inscricao,uf,regiao,tipo_escola,faixa_renda,nota_matematica
0,2023000000000000,SP,Sudeste,Pública,Média renda,470.1
1,2023000000000001,RR,Norte,Privada,Média-baixa renda,588.8
2,2023000000000002,RS,Sul,Pública,Média-baixa renda,704.4
4,2023000000000004,AP,Norte,Pública,Baixa renda,507.7
5,2023000000000005,GO,Centro-Oeste,Pública,Alta renda,451.0


In [7]:
def gold_nota_media_por_renda(df_silver: pd.DataFrame) -> pd.DataFrame:
    """Gold 1: nota média por área, agrupado por faixa de renda."""

    ordem_faixas = ['Baixa renda', 'Média-baixa renda', 'Média renda',
                     'Média-alta renda', 'Alta renda']

    agg = df_silver.groupby('faixa_renda').agg(
        qtd_participantes=('numero_inscricao', 'count'),
        nota_media_ciencias_natureza=('nota_ciencias_natureza', 'mean'),
        nota_media_ciencias_humanas=('nota_ciencias_humanas', 'mean'),
        nota_media_linguagens=('nota_linguagens', 'mean'),
        nota_media_matematica=('nota_matematica', 'mean'),
        nota_media_redacao=('nota_redacao', 'mean'),
    ).reset_index()

    # Nota geral média (média das 5 áreas)
    colunas_notas = [c for c in agg.columns if c.startswith('nota_media_')]
    agg['nota_media_geral'] = agg[colunas_notas].mean(axis=1)

    # Ordenar do menor pro maior poder aquisitivo (fica mais legível pro gráfico)
    agg['faixa_renda'] = pd.Categorical(agg['faixa_renda'], categories=ordem_faixas, ordered=True)
    agg = agg.sort_values('faixa_renda').reset_index(drop=True)

    # Arredondar pra facilitar leitura
    for col in colunas_notas + ['nota_media_geral']:
        agg[col] = agg[col].round(1)

    return agg


gold_renda = gold_nota_media_por_renda(df_silver)
gold_renda


,faixa_renda,qtd_participantes,nota_media_ciencias_natureza,nota_media_ciencias_humanas,nota_media_linguagens,nota_media_matematica,nota_media_redacao,nota_media_geral
0,Baixa renda,7938,521.3,540.4,529.3,508.4,602.6,540.4
1,Média-baixa renda,10566,521.0,540.2,530.5,508.2,599.2,539.8
2,Média renda,10584,519.7,540.3,529.8,509.4,598.9,539.6
3,Média-alta renda,8007,521.2,540.2,530.1,509.2,600.6,540.3
4,Alta renda,7946,518.9,539.2,530.6,509.7,598.3,539.4


In [8]:
def gold_publica_vs_privada_uf(df_silver: pd.DataFrame) -> pd.DataFrame:
    """Gold 2: comparação pública x privada por UF."""

    agg = df_silver.groupby(['uf', 'regiao', 'tipo_escola']).agg(
        qtd_participantes=('numero_inscricao', 'count'),
        nota_media_matematica=('nota_matematica', 'mean'),
        nota_media_redacao=('nota_redacao', 'mean'),
    ).reset_index()

    colunas_notas = ['nota_media_matematica', 'nota_media_redacao']
    agg['nota_media_geral'] = agg[colunas_notas].mean(axis=1)
    for col in colunas_notas + ['nota_media_geral']:
        agg[col] = agg[col].round(1)

    # Pivotar: uma linha por UF, colunas separadas pra pública e privada
    pivot = agg.pivot_table(
        index=['uf', 'regiao'],
        columns='tipo_escola',
        values='nota_media_geral'
    ).reset_index()

    pivot.columns.name = None
    pivot = pivot.rename(columns={'Pública': 'nota_media_publica', 'Privada': 'nota_media_privada'})

    # Gap = quanto a privada supera a pública (métrica central de desigualdade aqui)
    pivot['gap_privada_publica'] = (pivot['nota_media_privada'] - pivot['nota_media_publica']).round(1)

    pivot = pivot.sort_values('gap_privada_publica', ascending=False).reset_index(drop=True)

    return pivot


gold_uf = gold_publica_vs_privada_uf(df_silver)
gold_uf

,uf,regiao,nota_media_privada,nota_media_publica,gap_privada_publica
0,PR,Sul,570.2,550.3,19.9
1,RJ,Sudeste,574.0,556.3,17.7
2,CE,Nordeste,562.9,554.9,8.0
3,RN,Nordeste,560.7,553.9,6.8
4,BA,Nordeste,559.0,552.5,6.5
5,DF,Centro-Oeste,556.4,550.3,6.1
6,PB,Nordeste,557.8,553.7,4.1
7,PE,Nordeste,556.5,554.4,2.1
8,ES,Sudeste,550.1,552.4,-2.3
9,SP,Sudeste,550.2,552.6,-2.4


In [9]:
anos_disponiveis = [2021, 2022, 2023]
dfs_silver_por_ano = {2023: df_silver}  # já temos 2023 pronto

for ano_loop in [2021, 2022]:
    print(f"\n--- Processando ano {ano_loop} ---")

    # Bronze
    df_raw_ano = gerar_dados_sinteticos_enem(n=50_000, ano=ano_loop)
    df_bronze_ano = ingestao_bronze(df_raw_ano, fonte='sintetico_dev')

    caminho_bronze_ano = f'{BASE_PATH}/data/bronze/ano={ano_loop}'
    os.makedirs(caminho_bronze_ano, exist_ok=True)
    df_bronze_ano.to_parquet(f'{caminho_bronze_ano}/bronze.parquet', index=False)

    # Silver
    df_silver_ano = transformar_silver(df_bronze_ano)

    caminho_silver_ano = f'{BASE_PATH}/data/silver/ano={ano_loop}'
    os.makedirs(caminho_silver_ano, exist_ok=True)
    df_silver_ano.to_parquet(f'{caminho_silver_ano}/silver.parquet', index=False)

    dfs_silver_por_ano[ano_loop] = df_silver_ano

# Consolida tudo num único DataFrame (a Gold sempre olha pra Silver completa, todos os anos)
df_silver_todos_anos = pd.concat(dfs_silver_por_ano.values(), ignore_index=True)
print(f"\nTotal consolidado: {len(df_silver_todos_anos)} linhas, anos: {sorted(df_silver_todos_anos['ano'].unique())}")



--- Processando ano 2021 ---
Silver: 50000 → 44976 linhas (90.0% retido)

--- Processando ano 2022 ---
Silver: 50000 → 44956 linhas (89.9% retido)

Total consolidado: 134973 linhas, anos: [np.int64(2021), np.int64(2022), np.int64(2023)]


In [10]:
def gold_evolucao_gap_anual(df_silver_todos_anos: pd.DataFrame) -> pd.DataFrame:
    """Gold 3: evolução do gap de desempenho (privada-pública e alta-baixa renda) ao longo dos anos."""

    resultados = []

    for ano_iter, grupo_ano in df_silver_todos_anos.groupby('ano'):
        # Gap tipo de escola
        media_publica = grupo_ano.loc[grupo_ano['tipo_escola'] == 'Pública', 'nota_matematica'].mean()
        media_privada = grupo_ano.loc[grupo_ano['tipo_escola'] == 'Privada', 'nota_matematica'].mean()

        # Gap renda (alta vs baixa)
        media_renda_alta = grupo_ano.loc[grupo_ano['faixa_renda'] == 'Alta renda', 'nota_matematica'].mean()
        media_renda_baixa = grupo_ano.loc[grupo_ano['faixa_renda'] == 'Baixa renda', 'nota_matematica'].mean()

        resultados.append({
            'ano': ano_iter,
            'qtd_participantes': len(grupo_ano),
            'nota_media_publica': round(media_publica, 1),
            'nota_media_privada': round(media_privada, 1),
            'gap_escola_privada_publica': round(media_privada - media_publica, 1),
            'nota_media_renda_baixa': round(media_renda_baixa, 1),
            'nota_media_renda_alta': round(media_renda_alta, 1),
            'gap_renda_alta_baixa': round(media_renda_alta - media_renda_baixa, 1),
        })

    df_evolucao = pd.DataFrame(resultados).sort_values('ano').reset_index(drop=True)
    return df_evolucao


gold_evolucao = gold_evolucao_gap_anual(df_silver_todos_anos)
gold_evolucao

,ano,qtd_participantes,nota_media_publica,nota_media_privada,gap_escola_privada_publica,nota_media_renda_baixa,nota_media_renda_alta,gap_renda_alta_baixa
0,2021,44976,509.8,511.4,1.6,511.7,508.4,-3.3
1,2022,44956,510.5,508.9,-1.6,508.7,509.4,0.6
2,2023,45041,509.1,508.0,-1.1,508.4,509.7,1.3


In [11]:
import os

GOLD_PATH = os.path.join(BASE_PATH, 'data', 'gold')

# Salva cada tabela Gold como um arquivo Parquet único
gold_nota_media_por_renda.to_parquet(
    os.path.join(GOLD_PATH, 'gold_nota_media_por_renda.parquet'),
    index=False
)

gold_publica_vs_privada_uf.to_parquet(
    os.path.join(GOLD_PATH, 'gold_publica_vs_privada_uf.parquet'),
    index=False
)

gold_evolucao_gap_anual.to_parquet(
    os.path.join(GOLD_PATH, 'gold_evolucao_gap_anual.parquet'),
    index=False
)

# Confirmação
for arquivo in os.listdir(GOLD_PATH):
    caminho = os.path.join(GOLD_PATH, arquivo)
    tamanho_kb = os.path.getsize(caminho) / 1024
    print(f"✅ {arquivo} — {tamanho_kb:.1f} KB")

AttributeError: 'function' object has no attribute 'to_parquet'

In [12]:
print(type(gold_nota_media_por_renda))
print(type(gold_publica_vs_privada_uf))
print(type(gold_evolucao_gap_anual))

<class 'function'>
<class 'function'>
<class 'function'>


In [13]:
gold_nota_media_por_renda = gold_nota_media_por_renda(df_silver_todos_anos)  # ajusta o argumento pro que sua função pedir

In [14]:
# Detetive: acha todos os DataFrames que existem na memória agora
for nome, valor in list(globals().items()):
    if isinstance(valor, pd.DataFrame) and not nome.startswith('_'):
        print(f"{nome:35s} -> {len(valor)} linhas, colunas: {list(valor.columns)[:5]}...")

df_raw                              -> 50000 linhas, colunas: ['NU_INSCRICAO', 'NU_ANO', 'SG_UF_PROVA', 'TP_ESCOLA', 'Q006']...
df_bronze                           -> 50000 linhas, colunas: ['NU_INSCRICAO', 'NU_ANO', 'SG_UF_PROVA', 'TP_ESCOLA', 'Q006']...
df_silver                           -> 45041 linhas, colunas: ['numero_inscricao', 'ano', 'uf', 'tipo_escola_codigo', 'renda_codigo']...
gold_nota_media_por_renda           -> 5 linhas, colunas: ['faixa_renda', 'qtd_participantes', 'nota_media_ciencias_natureza', 'nota_media_ciencias_humanas', 'nota_media_linguagens']...
gold_renda                          -> 5 linhas, colunas: ['faixa_renda', 'qtd_participantes', 'nota_media_ciencias_natureza', 'nota_media_ciencias_humanas', 'nota_media_linguagens']...
gold_uf                             -> 27 linhas, colunas: ['uf', 'regiao', 'nota_media_privada', 'nota_media_publica', 'gap_privada_publica']...
df_raw_ano                          -> 50000 linhas, colunas: ['NU_INSCRICAO', 'NU_ANO', 

In [15]:
import os

GOLD_PATH = os.path.join(BASE_PATH, 'data', 'gold')

gold_renda.to_parquet(
    os.path.join(GOLD_PATH, 'gold_nota_media_por_renda.parquet'),
    index=False
)

gold_uf.to_parquet(
    os.path.join(GOLD_PATH, 'gold_publica_vs_privada_uf.parquet'),
    index=False
)

gold_evolucao.to_parquet(
    os.path.join(GOLD_PATH, 'gold_evolucao_gap_anual.parquet'),
    index=False
)

# Confirmação
for arquivo in sorted(os.listdir(GOLD_PATH)):
    caminho = os.path.join(GOLD_PATH, arquivo)
    tamanho_kb = os.path.getsize(caminho) / 1024
    print(f"✅ {arquivo} — {tamanho_kb:.1f} KB")

✅ gold_evolucao_gap_anual.parquet — 5.7 KB
✅ gold_nota_media_por_renda.parquet — 6.1 KB
✅ gold_publica_vs_privada_uf.parquet — 4.0 KB


In [17]:
from google.colab import userdata

SUPABASE_URI = userdata.get('SUPABASE_URI')

# Teste sem expor a senha: só confirma que carregou e mostra o formato
print(f"Secret carregado: {'✅ Sim' if SUPABASE_URI else '❌ Não'}")
print(f"Começa com: {SUPABASE_URI[:20]}..." if SUPABASE_URI else "")

TimeoutException: Requesting secret SUPABASE_URI timed out. Secrets can only be fetched when running from the Colab UI.

In [18]:
from google.colab import userdata

SUPABASE_URI = userdata.get('SUPABASE_URI')

print(f"Secret carregado: {'✅ Sim' if SUPABASE_URI else '❌ Não'}")
print(f"Começa com: {SUPABASE_URI[:20]}..." if SUPABASE_URI else "")

Secret carregado: ✅ Sim
Começa com: postgresql://postgre...


In [19]:
!pip install psycopg2-binary -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 41.5 MB/s eta 0:00:00


In [20]:
from sqlalchemy import create_engine, text

engine = create_engine(SUPABASE_URI)

try:
    with engine.connect() as conn:
        resultado = conn.execute(text("SELECT version();"))
        versao_postgres = resultado.fetchone()[0]
        print("✅ Conexão bem-sucedida!")
        print(f"Versão do Postgres: {versao_postgres}")
except Exception as e:
    print(f"❌ Erro na conexão: {e}")

❌ Erro na conexão: (psycopg2.OperationalError) connection to server at "db.apmadsqmncipjdqssply.supabase.co" (2600:1f14:359d:9301:e68d:4001:7a3b:9b0e), port 5432 failed: Network is unreachable
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [21]:
from google.colab import userdata
from sqlalchemy import create_engine, text

SUPABASE_URI = userdata.get('SUPABASE_URI')  # busca o valor atualizado

engine = create_engine(SUPABASE_URI)

try:
    with engine.connect() as conn:
        resultado = conn.execute(text("SELECT version();"))
        versao_postgres = resultado.fetchone()[0]
        print("✅ Conexão bem-sucedida!")
        print(f"Versão do Postgres: {versao_postgres}")
except Exception as e:
    print(f"❌ Erro na conexão: {e}")

❌ Erro na conexão: (psycopg2.OperationalError) connection to server at "db.apmadsqmncipjdqssply.supabase.co" (2600:1f14:359d:9301:e68d:4001:7a3b:9b0e), port 5432 failed: Network is unreachable
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [22]:
!pip install duckdb -q

In [23]:
import duckdb

# Conecta a uma "sessão" DuckDB em memória (sem arquivo, sem servidor)
con = duckdb.connect()

# Consulta direta no arquivo Parquet, sem precisar importar antes
resultado = con.execute(f"""
    SELECT *
    FROM read_parquet('{GOLD_PATH}/gold_nota_media_por_renda.parquet')
""").df()

print(resultado)

         faixa_renda  qtd_participantes  nota_media_ciencias_natureza  \
0        Baixa renda               7938                         521.3   
1  Média-baixa renda              10566                         521.0   
2        Média renda              10584                         519.7   
3   Média-alta renda               8007                         521.2   
4         Alta renda               7946                         518.9   

   nota_media_ciencias_humanas  nota_media_linguagens  nota_media_matematica  \
0                        540.4                  529.3                  508.4   
1                        540.2                  530.5                  508.2   
2                        540.3                  529.8                  509.4   
3                        540.2                  530.1                  509.2   
4                        539.2                  530.6                  509.7   

   nota_media_redacao  nota_media_geral  
0               602.6             540.

In [24]:
resultado_gap_renda = con.execute(f"""
    WITH base AS (
        SELECT
            faixa_renda,
            qtd_participantes,
            nota_media_geral,
            nota_media_matematica,
            nota_media_redacao
        FROM read_parquet('{GOLD_PATH}/gold_nota_media_por_renda.parquet')
    )
    SELECT
        faixa_renda,
        qtd_participantes,
        nota_media_geral,
        RANK() OVER (ORDER BY nota_media_geral DESC) AS ranking_desempenho,
        ROUND(nota_media_geral - LAG(nota_media_geral) OVER (ORDER BY faixa_renda), 2) AS variacao_vs_faixa_anterior,
        ROUND(nota_media_geral - FIRST_VALUE(nota_media_geral) OVER (ORDER BY faixa_renda), 2) AS gap_vs_faixa_mais_baixa,
        ROUND(100.0 * qtd_participantes / SUM(qtd_participantes) OVER (), 1) AS pct_do_total_participantes
    FROM base
    ORDER BY faixa_renda
""").df()

print(resultado_gap_renda)

         faixa_renda  qtd_participantes  nota_media_geral  ranking_desempenho  \
0         Alta renda               7946             539.4                   5   
1        Baixa renda               7938             540.4                   1   
2        Média renda              10584             539.6                   4   
3   Média-alta renda               8007             540.3                   2   
4  Média-baixa renda              10566             539.8                   3   

   variacao_vs_faixa_anterior  gap_vs_faixa_mais_baixa  \
0                         NaN                      0.0   
1                         1.0                      1.0   
2                        -0.8                      0.2   
3                         0.7                      0.9   
4                        -0.5                      0.4   

   pct_do_total_participantes  
0                        17.6  
1                        17.6  
2                        23.5  
3                        17.8  
4   

In [25]:
resultado_gap_renda = con.execute(f"""
    WITH base AS (
        SELECT
            faixa_renda,
            qtd_participantes,
            nota_media_geral,
            CASE faixa_renda
                WHEN 'Baixa renda' THEN 1
                WHEN 'Média-baixa renda' THEN 2
                WHEN 'Média renda' THEN 3
                WHEN 'Média-alta renda' THEN 4
                WHEN 'Alta renda' THEN 5
            END AS ordem_renda
        FROM read_parquet('{GOLD_PATH}/gold_nota_media_por_renda.parquet')
    )
    SELECT
        faixa_renda,
        qtd_participantes,
        nota_media_geral,
        RANK() OVER (ORDER BY nota_media_geral DESC) AS ranking_desempenho,
        ROUND(nota_media_geral - LAG(nota_media_geral) OVER (ORDER BY ordem_renda), 2) AS variacao_vs_faixa_anterior,
        ROUND(nota_media_geral - FIRST_VALUE(nota_media_geral) OVER (ORDER BY ordem_renda), 2) AS gap_vs_faixa_mais_baixa,
        ROUND(100.0 * qtd_participantes / SUM(qtd_participantes) OVER (), 1) AS pct_do_total_participantes
    FROM base
    ORDER BY ordem_renda
""").df()

print(resultado_gap_renda)

         faixa_renda  qtd_participantes  nota_media_geral  ranking_desempenho  \
0        Baixa renda               7938             540.4                   1   
1  Média-baixa renda              10566             539.8                   3   
2        Média renda              10584             539.6                   4   
3   Média-alta renda               8007             540.3                   2   
4         Alta renda               7946             539.4                   5   

   variacao_vs_faixa_anterior  gap_vs_faixa_mais_baixa  \
0                         NaN                      0.0   
1                        -0.6                     -0.6   
2                        -0.2                     -0.8   
3                         0.7                     -0.1   
4                        -0.9                     -1.0   

   pct_do_total_participantes  
0                        17.6  
1                        23.5  
2                        23.5  
3                        17.8  
4   

In [26]:
resultado_uf = con.execute(f"""
    WITH base AS (
        SELECT *
        FROM read_parquet('{GOLD_PATH}/gold_publica_vs_privada_uf.parquet')
    )
    SELECT
        uf,
        regiao,
        nota_media_publica,
        nota_media_privada,
        gap_privada_publica,
        RANK() OVER (ORDER BY gap_privada_publica DESC) AS ranking_gap_nacional,
        RANK() OVER (PARTITION BY regiao ORDER BY gap_privada_publica DESC) AS ranking_gap_na_regiao,
        ROUND(AVG(gap_privada_publica) OVER (PARTITION BY regiao), 2) AS gap_medio_da_regiao,
        ROUND(gap_privada_publica - AVG(gap_privada_publica) OVER (), 2) AS desvio_vs_media_nacional
    FROM base
    ORDER BY ranking_gap_nacional
""").df()

print(resultado_uf.head(10))

   uf        regiao  nota_media_publica  nota_media_privada  \
0  PR           Sul               550.3               570.2   
1  RJ       Sudeste               556.3               574.0   
2  CE      Nordeste               554.9               562.9   
3  RN      Nordeste               553.9               560.7   
4  BA      Nordeste               552.5               559.0   
5  DF  Centro-Oeste               550.3               556.4   
6  PB      Nordeste               553.7               557.8   
7  PE      Nordeste               554.4               556.5   
8  ES       Sudeste               552.4               550.1   
9  SP       Sudeste               552.6               550.2   

   gap_privada_publica  ranking_gap_nacional  ranking_gap_na_regiao  \
0                 19.9                     1                      1   
1                 17.7                     2                      1   
2                  8.0                     3                      1   
3                  6.8

In [3]:
query2 = """
SELECT
    uf,
    regiao,
    nota_media_publica,
    nota_media_privada,
    gap_privada_publica,

    -- Ranking nacional: quem tem o maior gap privada-pública no Brasil todo
    RANK() OVER (ORDER BY gap_privada_publica DESC) AS ranking_nacional,

    -- Ranking dentro da própria região
    RANK() OVER (PARTITION BY regiao ORDER BY gap_privada_publica DESC) AS ranking_regional,

    -- Gap médio da região (mesmo valor repetido pra todo estado da região)
    ROUND(AVG(gap_privada_publica) OVER (PARTITION BY regiao), 1) AS gap_medio_regiao,

    -- Quanto esse estado se desvia da média NACIONAL (não da região)
    ROUND(gap_privada_publica - AVG(gap_privada_publica) OVER (), 1) AS desvio_vs_media_nacional

FROM read_parquet('/content/drive/MyDrive/enem-medallion-etl/data/gold/gold_publica_vs_privada_uf.parquet')
ORDER BY ranking_nacional
"""

resultado_query2 = con.execute(query2).df()
resultado_query2

IOException: IO Error: No files found that match the pattern "/content/drive/MyDrive/enem-medallion-etl/data/gold/gold_publica_vs_privada_uf.parquet"

In [2]:
import duckdb
con = duckdb.connect()

In [4]:
query2 = """
SELECT
    uf,
    regiao,
    nota_media_publica,
    nota_media_privada,
    gap_privada_publica,

    -- Ranking nacional: quem tem o maior gap privada-pública no Brasil todo
    RANK() OVER (ORDER BY gap_privada_publica DESC) AS ranking_nacional,

    -- Ranking dentro da própria região
    RANK() OVER (PARTITION BY regiao ORDER BY gap_privada_publica DESC) AS ranking_regional,

    -- Gap médio da região (mesmo valor repetido pra todo estado da região)
    ROUND(AVG(gap_privada_publica) OVER (PARTITION BY regiao), 1) AS gap_medio_regiao,

    -- Quanto esse estado se desvia da média NACIONAL (não da região)
    ROUND(gap_privada_publica - AVG(gap_privada_publica) OVER (), 1) AS desvio_vs_media_nacional

FROM read_parquet('/content/drive/MyDrive/enem-medallion-etl/data/gold/gold_publica_vs_privada_uf.parquet')
ORDER BY ranking_nacional
"""

resultado_query2 = con.execute(query2).df()
resultado_query2

IOException: IO Error: No files found that match the pattern "/content/drive/MyDrive/enem-medallion-etl/data/gold/gold_publica_vs_privada_uf.parquet"

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
query2 = """
SELECT
    uf,
    regiao,
    nota_media_publica,
    nota_media_privada,
    gap_privada_publica,

    -- Ranking nacional: quem tem o maior gap privada-pública no Brasil todo
    RANK() OVER (ORDER BY gap_privada_publica DESC) AS ranking_nacional,

    -- Ranking dentro da própria região
    RANK() OVER (PARTITION BY regiao ORDER BY gap_privada_publica DESC) AS ranking_regional,

    -- Gap médio da região (mesmo valor repetido pra todo estado da região)
    ROUND(AVG(gap_privada_publica) OVER (PARTITION BY regiao), 1) AS gap_medio_regiao,

    -- Quanto esse estado se desvia da média NACIONAL (não da região)
    ROUND(gap_privada_publica - AVG(gap_privada_publica) OVER (), 1) AS desvio_vs_media_nacional

FROM read_parquet('/content/drive/MyDrive/enem-medallion-etl/data/gold/gold_publica_vs_privada_uf.parquet')
ORDER BY ranking_nacional
"""

resultado_query2 = con.execute(query2).df()
resultado_query2

,uf,regiao,nota_media_publica,nota_media_privada,gap_privada_publica,ranking_nacional,ranking_regional,gap_medio_regiao,desvio_vs_media_nacional
0,PR,Sul,550.3,570.2,19.9,1,1,2.6,23.3
1,RJ,Sudeste,556.3,574.0,17.7,2,1,0.9,21.1
2,CE,Nordeste,554.9,562.9,8.0,3,1,-0.7,11.4
3,RN,Nordeste,553.9,560.7,6.8,4,2,-0.7,10.2
4,BA,Nordeste,552.5,559.0,6.5,5,3,-0.7,9.9
5,DF,Centro-Oeste,550.3,556.4,6.1,6,1,-5.8,9.5
6,PB,Nordeste,553.7,557.8,4.1,7,4,-0.7,7.5
7,PE,Nordeste,554.4,556.5,2.1,8,5,-0.7,5.5
8,ES,Sudeste,552.4,550.1,-2.3,9,2,0.9,1.1
9,SP,Sudeste,552.6,550.2,-2.4,10,3,0.9,1.0


In [7]:
query3 = """
SELECT
    ano,
    qtd_participantes,
    nota_media_publica,
    nota_media_privada,
    gap_escola_privada_publica,
    nota_media_renda_baixa,
    nota_media_renda_alta,
    gap_renda_alta_baixa,

    -- Variação do gap escola em relação ao ano anterior
    gap_escola_privada_publica - LAG(gap_escola_privada_publica) OVER (ORDER BY ano)
        AS variacao_gap_escola_vs_ano_anterior,

    -- Variação do gap renda em relação ao ano anterior
    gap_renda_alta_baixa - LAG(gap_renda_alta_baixa) OVER (ORDER BY ano)
        AS variacao_gap_renda_vs_ano_anterior,

    -- Gap escola do primeiro ano disponível, repetido em toda linha (baseline pra comparar)
    FIRST_VALUE(gap_escola_privada_publica) OVER (ORDER BY ano)
        AS gap_escola_ano_base,

    -- Quanto o gap escola mudou desde o primeiro ano até agora (acumulado)
    gap_escola_privada_publica - FIRST_VALUE(gap_escola_privada_publica) OVER (ORDER BY ano)
        AS variacao_acumulada_desde_ano_base

FROM read_parquet('/content/drive/MyDrive/enem-medallion-etl/data/gold/gold_evolucao_gap_anual.parquet')
ORDER BY ano
"""

resultado_query3 = con.execute(query3).df()
resultado_query3

,ano,qtd_participantes,nota_media_publica,nota_media_privada,gap_escola_privada_publica,nota_media_renda_baixa,nota_media_renda_alta,gap_renda_alta_baixa,variacao_gap_escola_vs_ano_anterior,variacao_gap_renda_vs_ano_anterior,gap_escola_ano_base,variacao_acumulada_desde_ano_base
0,2021,44976,509.8,511.4,1.6,511.7,508.4,-3.3,NaN,NaN,1.6,0.0
1,2022,44956,510.5,508.9,-1.6,508.7,509.4,0.6,-3.2,3.9,1.6,-3.2
2,2023,45041,509.1,508.0,-1.1,508.4,509.7,1.3,0.5,0.7,1.6,-2.7
